# Import libraries

In [ ]:
import os
import sys
import time
import datetime

from termcolor import colored, cprint

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import sys
from os import listdir

import numpy as np
import pandas as pd  # Import Pandas and Numpy to create databases
import tensorflow as tf  # Import tensorflow to do tensor manipulations
from abyss.helpers import metrics_viewer
from abyss.models import create_decoder
from abyss.glob import glob
import statsmodels.api as sm
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
from sklearn.metrics import roc_curve, auc

phenotypes_labels = {
    "bc": "Breast Cancer",
    "cad": "Coronary Artery Disease",
    "ms": "Multiple Sclerosis",
    "t2d": "Type 2 Diabetes",
    "pc": "Prostate Cancer",
    "ibd": "Inflammatory Bowel Disease"
}

scenario_labels = {
    "no_cov": "no covariates",
    "abyss": "Abyss correction",
    "cov": "PCA correction",
    "abyss_cov": "Abyss + PCA as covariates"
}

In [ ]:
path_ld_scores = f"{glob.PATH_usefull}/eur_w_ld_chr.tar.gz"

In [ ]:
import tarfile

In [ ]:
# Open the tar.gz file
with tarfile.open(path_ld_scores, "r:gz") as tar:
    # Extract all contents to the specified directory
    tar.extractall(path=glob.PATH_usefull)

    # Assuming there's only one file in the tar.gz, get its name
    file_name = tar.getnames()[0]

In [ ]:
file_name

In [ ]:
# Directory path
dir_path = f"{glob.PATH_usefull}/{file_name}"

# List all files in the directory
file_list = os.listdir(dir_path)
files = [f for f in file_list if f.split('.')[-1] == 'gz']

In [ ]:
ldscore_dfs = []
for file in files:
    # Example for reading a .gz file
    gz_file = os.path.join(dir_path, file)
    df_gz = pd.read_csv(gz_file, compression='gzip',sep='\t')
    ldscore_dfs.append(df_gz)

In [ ]:
combined_df = pd.concat(ldscore_dfs, ignore_index=True)

In [ ]:
combined_df.rename(columns={'BP': 'POS'}, inplace=True)
combined_df

# Extract sumstats

In [ ]:
disease = "cad"
scenario = "no_cov"

In [ ]:
path_input = glob.PATH_GENO + f"/09_summary_statistics/{disease}/{scenario}"

In [ ]:
list_of_bins = os.listdir(path_input)

In [ ]:
list_of_dfs = []
for biny in list_of_bins:
    try:
        temp = pd.read_pickle(path_input+f"/{biny}")
        temp['bin'] = biny.split('_')[1]
        
        # Sort by -logp in descending order and take the top 1000 SNPs
        top_snps = temp.sort_values(by='-logp', ascending=False)
        
        list_of_dfs.append(top_snps)
    except Exception as e:
        print(e)
        print(f"Problem with chrom {biny}")

# Concatenating the top SNPs from each bin
sumstat = pd.concat(list_of_dfs)

# Sorting by chromosome if needed
sumstat = sumstat.sort_values(by='#CHROM', ascending=True)

In [ ]:
sumstat

In [ ]:
merged_df = pd.merge(sumstat, combined_df, on='POS')

In [ ]:
path_pheno = glob.PATH_PHENO
pheno = pd.read_pickle(f"{path_pheno}/{disease}.pkl")
N = len(pheno)
M = len(merged_df)

In [ ]:
merged_df['X2'] = (merged_df['beta_snp']**2/merged_df['std_err']**2)**(1/2)

In [ ]:
merged_df['LDscore'] = merged_df['L2']

In [ ]:
merged_df

In [ ]:
merged_df['weights'] = 1/(1+merged_df['LDscore'])
merged_df['bin'] = merged_df['bin'].astype(int)

In [ ]:
#merged_df = merged_df[merged_df['-logp']< 5] # filter out ultra significant snps
merged_df = merged_df[merged_df['-logp']> 8] # filter out non significant snps
#merged_df = merged_df[merged_df['bin'] > 1] # filter out ultra rare variants
#merged_df = merged_df[merged_df['bin'] < 32] # filter out for palindromic snps
#merged_df = merged_df[merged_df['std_err'] < 0.1] # filter out for too high standard error
merged_df = merged_df[merged_df['LDscore'] < 40]
#merged_df = merged_df[merged_df['X2'] > 1]
merged_df

In [ ]:
# LD Score Regression
X = merged_df['LDscore']
y = merged_df['X2']
weights = merged_df['weights']
# Add a constant to the model (intercept)
X = sm.add_constant(X)

# Weighted least squares regression
model = sm.WLS(y, X, weights=weights)
results = model.fit()

In [ ]:
X = merged_df['LDscore']
y = merged_df['X2']
weights = merged_df['weights']

# Create a scatterplot
scatter = sns.scatterplot(x=X, y=y, hue=weights, palette='rocket')

# Overlay the regression line
sns.regplot(x=X, y=y, scatter=False, color='red', ci=None)

plt.xlabel('LD Score')
plt.ylabel('X2')
plt.title('Scatterplot with Regression Line')

# Place a legend on the plot
scatter.legend(title='Weights', loc='upper right')

plt.show()

In [ ]:
# Get the intercept from the model
intercept = results.params['const']

# Get the slope (coefficient for 'LD_score') from the model
slope = results.params['LDscore']
print(intercept)
print(slope)


# Bin together based on LD score

In [ ]:
 # Define the number of bins (100 in your case)
num_bins = 200

# Create the bins for LD scores
bins = pd.cut(merged_df['LDscore'], bins=num_bins)

# Group the data by the LD score bins and calculate the mean LD_score and X2 for each bin
mean_ld_x2 = merged_df.groupby(bins, observed=False).agg({'LDscore': 'mean', 'X2': 'mean','std_err':'mean'})

# Create a new DataFrame with mean LD scores and X2 for each LD score bin
new_df = pd.DataFrame({
    'LDscore': mean_ld_x2['LDscore'],
    'X2': mean_ld_x2['X2'],
    'std_err': mean_ld_x2['std_err'],
    'LD_score_bin': mean_ld_x2.index.categories
})

In [ ]:
weights = 1/(1+new_df['LDscore'])

# Add normalized weights to your DataFrame
new_df['weights'] = weights

In [ ]:
new_df = new_df.dropna(axis=0)

In [ ]:
X = new_df['LDscore']
y = new_df['X2']
weights = new_df['weights']
# Add a constant to the model (intercept)
X = sm.add_constant(X)

# Weighted least squares regression
model = sm.WLS(y, X, weights=weights)
results = model.fit()

In [ ]:

# Add a constant to the model (intercept)
X = sm.add_constant(X)

# Weighted least squares regression
model = sm.WLS(y, X, weights=weights)
results = model.fit()

# Create a scatterplot
scatter = sns.scatterplot(x=X['LDscore'], y=y, hue=weights, palette='rocket')

# Get the fitted values from the model
fitted_values = results.fittedvalues

# Overlay the regression line based on the model
plt.plot(X['LDscore'], fitted_values, color='red')

plt.xlabel('LD Score bin')
plt.ylabel('X2')
plt.title('Scatterplot with WLS Regression Line')

# Place a legend on the plot
scatter.legend(title='Weights', loc='upper right')

plt.show()

# Iterate over diseases and scenarios

In [ ]:
diseases = ['ms', 'cad', 'bc']

In [ ]:
scenarios = ['no_cov', 'cov', 'abyss_cov', 'abyss']

In [ ]:
bin_technique = 1

In [ ]:
# Define the number of bins
num_bins = 200

In [ ]:
confounders = []
heritabilities = []
tools = []
disease_an = []

# Without binning

In [ ]:
for disease in diseases:
    for scenario in scenarios:
        path_input = glob.PATH_GENO + f"/09_summary_statistics/{disease}/{scenario}"
        list_of_bins = os.listdir(path_input)
        list_of_dfs = []
        for biny in list_of_bins:
            try:
                temp = pd.read_pickle(path_input+f"/{biny}")
                temp['bin'] = biny.split('_')[1]
                
                # Sort by -logp in descending order and take the top 1000 SNPs
                top_snps = temp.sort_values(by='-logp', ascending=False)
                
                list_of_dfs.append(top_snps)
            except Exception as e:
                print(e)
                print(f"Problem with chrom {biny}")
        
        # Concatenating the top SNPs from each bin
        sumstat = pd.concat(list_of_dfs)
        
        # Sorting by chromosome if needed
        sumstat = sumstat.sort_values(by='#CHROM', ascending=True)

        merged_df = pd.merge(sumstat, combined_df, on='POS')

        path_pheno = glob.PATH_PHENO
        pheno = pd.read_pickle(f"{path_pheno}/{disease}.pkl")
        N = len(pheno)
        sumstat['ID'] = sumstat['snp'].str.rsplit('_', n=1).str[0]

        # Merging the DataFrames on the 'Key' column
        merged_df['LD_score'] = merged_df['L2']
        merged_df['bin'] = merged_df['bin'].astype(int)
        merged_df['X2'] = ((merged_df['beta_snp'] ** 2) / (merged_df['std_err'])**2)
        merged_df['weights'] = 1/(1+merged_df['LD_score'])

        merged_df = merged_df[merged_df['-logp']> 0.1] # filter out ultra significant snps
        merged_df = merged_df[merged_df['-logp']< 16] # filter out non significant snps
        #merged_df = merged_df[merged_df['bin'] > 1] # filter out ultra rare variants
        #merged_df = merged_df[merged_df['bin'] < 32] # filter out for palindromic snps
        #merged_df = merged_df[merged_df['std_err'] < 0.1] # filter out for too high standard error
        merged_df = merged_df[merged_df['LD_score'] < 50]
        #merged_df = merged_df[merged_df['X2'] > 1]
        M = len(merged_df)
                    
        # LD Score Regression
        X = merged_df['LD_score']
        y = merged_df['X2']
        weights = merged_df['weights']
        # Add a constant to the model (intercept)
        X = sm.add_constant(X)
        
        # Weighted least squares regression
        model = sm.WLS(y, X, weights=weights)
        results = model.fit()


        X = merged_df['LD_score']
        weights = merged_df['weights']
        
        # Create a scatterplot
        scatter = sns.scatterplot(x=X, y=y, hue=weights, palette='rocket')
        
        # Overlay the regression line
        sns.regplot(x=X, y=y, scatter=False, color='red', ci=None)
        
        plt.xlabel('LD Score')
        plt.ylabel('X2')
        plt.title('Scatterplot with Regression Line')
        
        # Place a legend on the plot
        scatter.legend(title='Weights', loc='upper right')
        
        plt.show()

        # Get the intercept from the model
        intercept = results.params['const']
        
        # Get the slope (coefficient for 'LD_score') from the model
        slope = results.params['LD_score']
        
        print(f'Intercept: {intercept}')
        print(f'Slope: {slope}')

        effect_confounders = (intercept - 1)
        heritability_estimated = slope/M

        confounders.append(effect_confounders)
        heritabilities.append(heritability_estimated)
        tools.append(scenario)
        disease_an.append(disease)

In [ ]:
ld_dataframe = pd.DataFrame({
    'Disease': disease_an,
    'Scenario': tools,
    'heritability_LD': heritabilities,
    'confounders': confounders
})

In [ ]:
ld_dataframe

In [ ]:
# Create a bar plot
sns.barplot(x='Scenario', y='confounders', data=ld_dataframe, hue='Disease')

# Adding labels and title
plt.xlabel('Tools')
plt.ylabel('Confounders')
plt.title('Intercept Values by Scenario')

# Show the plot
# Show the plot
plt.show()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Find the indices of the rows with the lowest confounder values within each 'bins' group
lowest_confounder_indices = ld_dataframe.groupby('Disease')['confounders'].idxmin()

# Create a new dataframe with only the rows corresponding to the lowest confounder values
lowest_confounder_df = ld_dataframe.loc[lowest_confounder_indices]

# Create a bar plot with the lowest confounder values
sns.barplot(x='Disease', y='confounders', data=lowest_confounder_df, hue='Scenario')

# Adding labels and title
plt.xlabel('Tools')
plt.ylabel('Confounders')
plt.title('Intercept Values by Scenario')

# Show the plot
plt.show()

# Check the performance per bin

In [ ]:
bins_AFs = list(range(33))[1:]

In [ ]:
diseases = ['cad','ms']
scenarios = ['abyss', 'no_cov', 'abyss_cov', 'cov']

confounders = []
heritabilities = []
tools = []
disease_an = []
binysss = []

In [ ]:
for biny in bins_AFs:
    for disease in diseases:
        for scenario in scenarios:
            path_input = glob.PATH_GENO + f"/09_summary_statistics/{disease}/{scenario}"
            temp = pd.read_pickle(path_input+f"/bin_{biny}_sumstat.pkl")
            temp['bin'] = biny
            # Sort by -logp in descending order and take the top 1000 SNPs
            sumstat = temp.sort_values(by='-logp', ascending=False) 
            
            # Sorting by chromosome if needed
            sumstat = sumstat.sort_values(by='#CHROM', ascending=True)
    
            merged_df = pd.merge(sumstat, combined_df, on='POS')
    
            path_pheno = glob.PATH_PHENO
            pheno = pd.read_pickle(f"{path_pheno}/{disease}.pkl")
            N = len(pheno)
            sumstat['ID'] = sumstat['snp'].str.rsplit('_', n=1).str[0]
    
            # Merging the DataFrames on the 'Key' column
            merged_df['LD_score'] = merged_df['L2']
            merged_df['bin'] = merged_df['bin'].astype(int)
            merged_df['X2'] = ((merged_df['beta_snp'] ** 2) / (merged_df['std_err'])**2)
            merged_df['weights'] = 1/(1+merged_df['LD_score'])
    
            merged_df = merged_df[merged_df['-logp']< 6] # filter out ultra significant snps
            merged_df = merged_df[merged_df['-logp']> 3] # filter out non significant snps
            #merged_df = merged_df[merged_df['bin'] > 1] # filter out ultra rare variants
            #merged_df = merged_df[merged_df['bin'] < 32] # filter out for palindromic snps
            #merged_df = merged_df[merged_df['std_err'] < 0.1] # filter out for too high standard error
            #merged_df = merged_df[merged_df['LD_score'] < 40]
            #merged_df = merged_df[merged_df['X2'] > 1]
            M = len(merged_df)
                        
            # LD Score Regression
            X = merged_df['LD_score']
            y = merged_df['X2']
            weights = merged_df['weights']
            # Add a constant to the model (intercept)
            X = sm.add_constant(X)
            
            # Weighted least squares regression
            model = sm.WLS(y, X, weights=weights)
            results = model.fit()
    
    
            X = merged_df['LD_score']
            weights = merged_df['weights']
            
            # Create a scatterplot
            scatter = sns.scatterplot(x=X, y=y, hue=weights, palette='rocket')
            
            # Overlay the regression line
            sns.regplot(x=X, y=y, scatter=False, color='red', ci=None)
            
            plt.xlabel('LD Score')
            plt.ylabel('X2')
            plt.title('Scatterplot with Regression Line')
            
            # Place a legend on the plot
            scatter.legend(title='Weights', loc='upper right')
            
            plt.show()
    
            # Get the intercept from the model
            intercept = results.params['const']
            
            # Get the slope (coefficient for 'LD_score') from the model
            slope = results.params['LD_score']
            
            print(f'Intercept: {intercept}')
            print(f'Slope: {slope}')
    
            effect_confounders = (intercept - 1)
            heritability_estimated = N*(slope/M)
    
            confounders.append(effect_confounders)
            heritabilities.append(heritability_estimated)
            tools.append(scenario)
            disease_an.append(disease)
            binysss.append(biny)

In [ ]:
ld_dataframe = pd.DataFrame({
    'Disease': disease_an,
    'Scenario': tools,
    'heritability_LD': heritabilities,
    'confounders': confounders,
    'bins':binysss
})

In [ ]:
ld_dataframe = ld_dataframe[ld_dataframe['confounders'] > 0]

In [ ]:
# Create a bar plot
sns.barplot(x='bins', y='confounders', data=ld_dataframe, hue='Scenario')

# Adding labels and title
plt.xlabel('Tools')
plt.ylabel('Confounders')
plt.title('Intercept Values by Scenario')
# Show the plot
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Find the indices of the rows with the lowest confounder values within each 'bins' group
lowest_confounder_indices = ld_dataframe.groupby('bins')['confounders'].idxmin()

# Create a new dataframe with only the rows corresponding to the lowest confounder values
lowest_confounder_df = ld_dataframe.loc[lowest_confounder_indices]

# Create a bar plot with the lowest confounder values
sns.barplot(x='bins', y='confounders', data=lowest_confounder_df, hue='Scenario')

# Adding labels and title
plt.xlabel('Tools')
plt.ylabel('Confounders')
plt.title('Intercept Values by Scenario')

# Show the plot
plt.show()